# 07_constrained_extraction

07_constrained_extraction.py — 스키마 제약으로 추출 품질 높이기

자유 추출 (06) 은 노드/관계 타입이 들쭉날쭉. 화이트리스트로 일관성↑.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '07_constrained_extraction.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
07_constrained_extraction.py — 스키마 제약으로 추출 품질 높이기

자유 추출 (06) 은 노드/관계 타입이 들쭉날쭉. 화이트리스트로 일관성↑.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

from _common import get_llm, banner, llm_unavailable
from importlib import import_module
SAMPLE_TEXT = import_module("06_llm_graph_transformer").SAMPLE_TEXT


def main() -> None:
    banner("LLMGraphTransformer — 스키마 제약 (allowed_nodes / allowed_relationships)")
    llm = get_llm()
    if llm is None:
        llm_unavailable()
        return

    transformer = LLMGraphTransformer(
        llm=llm,
        allowed_nodes=["Person", "Company", "Product"],
        allowed_relationships=["FOUNDED", "DEVELOPED", "INVESTED_IN", "WORKED_AT"],
        ignore_tool_usage=True,   # free 모델 호환
    )

    docs = [Document(page_content=SAMPLE_TEXT.strip())]
    graph_docs = transformer.convert_to_graph_documents(docs)
    gd = graph_docs[0]

    print(f"\n📦 노드 ({len(gd.nodes)}개) — type 은 {{Person|Company|Product}} 내로 제한")
    for n in gd.nodes:
        print(f"  - ({n.id}, type={n.type})")

    print(f"\n🔗 관계 ({len(gd.relationships)}개) — type 은 화이트리스트 내로 제한")
    for r in gd.relationships:
        print(f"  - ({r.source.id})-[:{r.type}]->({r.target.id})")


if __name__ == "__main__":
    main()

C:\Users\user\AppData\Local\Temp\ipykernel_14608\1319835320.py:11: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📌 LLMGraphTransformer — 스키마 제약 (allowed_nodes / allowed_relationships)



📦 노드 (6개) — type 은 {Person|Company|Product} 내로 제한
  - (OpenAI, type=Company)
  - (Claude, type=Product)
  - (Anthropic, type=Company)
  - (다니엘라 아모데이, type=Person)
  - (Amazon, type=Company)
  - (다리오 아모데이, type=Person)

🔗 관계 (5개) — type 은 화이트리스트 내로 제한
  - (다리오 아모데이)-[:FOUNDED]->(Anthropic)
  - (다니엘라 아모데이)-[:FOUNDED]->(Anthropic)
  - (Anthropic)-[:DEVELOPED]->(Claude)
  - (Amazon)-[:INVESTED_IN]->(Anthropic)
  - (다리오 아모데이)-[:WORKED_AT]->(OpenAI)
